# v2 — Monthly means of z, temp, sal (1993-2013 production run, all four seeds)

Maps of the **mean particle property** (depth, temperature, salinity) per spatial bin, grouped by
**release month**, over the 1534 zarr stores of the new `tracks_v3_*` production run
(4 seeds -> ONE 21-year climatology).

Method is the template's (`Analysis/Monthly_means_z_temp_sal.ipynb`): binned **sum of the variable**
divided by the binned **count of samples** = mean value per bin, per release month.
The only change is that the 1534 stores (~15.5M trajectories) are **streamed one by one**
with `xhistogram` (sums and counts accumulated in 12 x nlon x nlat arrays) instead of concatenated,
so memory stays flat: counts and weighted sums are additive, so the result is identical.
The release position (`obs=0`) is skipped, as in the template.

In [ ]:
# --- papermill parameters ---------------------------------------------------
tracks_root  = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2"
seeds        = ["11098723", "45678", "67891", "78876"]   # all four seeds -> one 1993-2013 climatology
out_dir      = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis"
fig_dir      = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis/figures"
bins_file    = "v2.bins_v3.npz"   # bin edges shared by both v2 notebooks (written by the z/T/S one)
n_lon_bins   = 80        # number of lon bin EDGES (as in the template)
n_lat_bins   = 30        # number of lat bin EDGES
n_scan       = 120       # stores sampled to fix the bin edges (only used if bins_file is absent)
n_jobs       = 16        # parallel processes streaming the stores
max_stores   = 0         # 0 = all stores; >0 = test mode on the first N stores
drop_first_obs = True    # skip obs=0 (the release position), exactly as in the template
sal_min      = 0.0       # samples with sal <= sal_min are treated as invalid
temp_min     = 0.0       # samples with temp <= temp_min are treated as invalid


In [ ]:
import os, time, json
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from xhistogram.xarray import histogram as xhist
from joblib import Parallel, delayed
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cf
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cmocean as cm

os.makedirs(out_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)
os.chdir(out_dir)
print("working in", os.getcwd())


## Stores

In [ ]:
stores = []
for s in seeds:
    ss = sorted((Path(tracks_root) / f"tracks_v3_{s}").glob("Parcels_run_*.zarr"))
    print(f"seed {s}: {len(ss)} stores")
    stores += ss
if max_stores:
    stores = stores[:max_stores]
    print("TEST MODE: using only", len(stores), "stores")
print("total stores:", len(stores))


## Bin edges
Fixed once (from a spread sample of stores, padded outward) so every store bins onto the same grid,
and so the buoyancy/sigma0 notebook can reuse exactly the same grid.

In [ ]:
def _lonlat_range(f):
    d = xr.open_zarr(f)
    lo = d.lon.values; la = d.lat.values
    return (np.nanmin(lo), np.nanmax(lo), np.nanmin(la), np.nanmax(la))

if os.path.exists(bins_file):
    z = np.load(bins_file)
    lon_bins, lat_bins = z["lon_bins"], z["lat_bins"]
    print("loaded bin edges from", bins_file)
else:
    idx = np.linspace(0, len(stores) - 1, min(n_scan, len(stores))).astype(int)
    t0 = time.time()
    r = Parallel(n_jobs=n_jobs)(delayed(_lonlat_range)(stores[i]) for i in idx)
    r = np.array(r)
    lo0, lo1, la0, la1 = r[:, 0].min(), r[:, 1].max(), r[:, 2].min(), r[:, 3].max()
    # pad outward by ~2 bin widths so particles outside the scanned sample still fall inside
    dlo = (lo1 - lo0) / (n_lon_bins - 1); dla = (la1 - la0) / (n_lat_bins - 1)
    lon_bins = np.linspace(lo0 - 2 * dlo, lo1 + 2 * dlo, n_lon_bins)
    lat_bins = np.linspace(la0 - 2 * dla, la1 + 2 * dla, n_lat_bins)
    np.savez(bins_file, lon_bins=lon_bins, lat_bins=lat_bins)
    print(f"scanned {len(idx)} stores in {time.time()-t0:.0f}s -> saved {bins_file}")

lon_c = 0.5 * (lon_bins[1:] + lon_bins[:-1])
lat_c = 0.5 * (lat_bins[1:] + lat_bins[:-1])
NLON, NLAT = len(lon_c), len(lat_c)
print(f"lon bins: {lon_bins[0]:.2f} .. {lon_bins[-1]:.2f}  ({NLON} bins)")
print(f"lat bins: {lat_bins[0]:.2f} .. {lat_bins[-1]:.2f}  ({NLAT} bins)")


## Streaming binned sum / count

In [ ]:
VARS = ["z", "temp", "sal"]

def _bin_one(f, lon_bins, lat_bins, vlist, drop_first_obs, sal_min, temp_min):
    """Stream ONE store: weighted xhistogram (binned sum) + binned count per variable.

    Exactly the template's estimator (Analysis/Monthly_means_z_temp_sal.ipynb):
        count = xhist(lon, lat, bins=[lon_bins, lat_bins], dim=["trajectory"])
        wsum  = xhist(lon, lat, bins=..., dim=["trajectory"], weights=var)
    with the coordinates masked where the variable is NaN, summed over obs.
    Counts and weighted sums are **additive**, so accumulating them store by store and
    dividing sum/count at the very end gives exactly the same monthly mean per bin as
    running the concatenated dataset in one go - but memory stays flat.
    Returns (month, sums, counts, stats) with tiny (NLON, NLAT) arrays.
    """
    ds = xr.open_zarr(f)
    month = int(pd.Timestamp(ds.time.values[0, 0]).month)
    sl = slice(1, None) if drop_first_obs else slice(None)   # skip obs=0 (release position)
    d = ds[["lon", "lat", "z", "temp", "sal"]].isel(obs=sl).compute()

    bad = (~np.isfinite(d.temp)) | (~np.isfinite(d.sal)) | (d.sal <= sal_min) | (d.temp <= temp_min)
    d["temp"] = d.temp.where(~bad)
    d["sal"]  = d.sal.where(~bad)

    fields = {"z": d.z, "temp": d.temp, "sal": d.sal}

    sums, counts = {}, {}
    stats = {"n_bad": int(bad.values.sum()),
             "n_finite_pos": int((np.isfinite(d.lon.values) & np.isfinite(d.lat.values)).sum()),
             "lon_min": float(d.lon.min()), "lon_max": float(d.lon.max()),
             "lat_min": float(d.lat.min()), "lat_max": float(d.lat.max()),
             "n_out": 0}
    for v in vlist:
        w   = fields[v]
        lon = d.lon.where(w.notnull())      # drop samples with no value for this var
        lat = d.lat.where(w.notnull())
        count = xhist(lon, lat, bins=[lon_bins, lat_bins], dim=["trajectory"], bin_dim_suffix="")
        wsum  = xhist(lon, lat, bins=[lon_bins, lat_bins], dim=["trajectory"], weights=w,
                      bin_dim_suffix="")
        counts[v] = count.sum("obs").transpose("lon", "lat").values
        sums[v]   = wsum.sum("obs").transpose("lon", "lat").values

        vals = w.values.ravel()
        vals = vals[np.isfinite(vals) & np.isfinite(lon.values.ravel())
                    & np.isfinite(lat.values.ravel())]
        stats["n_" + v]   = int(vals.size)
        stats["n_out"]    = max(stats["n_out"], int(vals.size - counts[v].sum()))
        stats["min_" + v] = float(vals.min()) if vals.size else np.nan
        stats["max_" + v] = float(vals.max()) if vals.size else np.nan
        stats["sum_" + v] = float(vals.sum())
    return month, sums, counts, stats


In [ ]:
SUM   = {v: np.zeros((12, NLON, NLAT)) for v in VARS}
COUNT = {v: np.zeros((12, NLON, NLAT)) for v in VARS}
all_stats = []

CHUNK = max(n_jobs * 4, 32)
t0 = time.time()
for i0 in range(0, len(stores), CHUNK):
    batch = stores[i0:i0 + CHUNK]
    res = Parallel(n_jobs=n_jobs)(
        delayed(_bin_one)(f, lon_bins, lat_bins, VARS, drop_first_obs, sal_min, temp_min)
        for f in batch)
    for month, sums, counts, st in res:
        for v in VARS:
            SUM[v][month - 1]   += sums[v]
            COUNT[v][month - 1] += counts[v]
        st["month"] = month
        all_stats.append(st)
    el = time.time() - t0
    done = i0 + len(batch)
    print(f"{done}/{len(stores)} stores  {el/60:.1f} min elapsed, "
          f"ETA {(el/done*(len(stores)-done))/60:.1f} min", flush=True)
print(f"streaming finished in {(time.time()-t0)/60:.1f} min")
stats = pd.DataFrame(all_stats)


## Sanity checks

In [ ]:
print("stores processed per release month:")
print(stats.groupby("month").size().to_string())
print()
print(f"total position samples (obs>0): {stats.n_finite_pos.sum():,.0f}")
print(f"samples falling OUTSIDE the lon/lat bins: {stats.n_out.sum():,.0f}")
print(f"samples rejected as invalid (sal<={sal_min} or temp<={temp_min} or NaN): {stats.n_bad.sum():,.0f}")
print(f"observed lon range {stats.lon_min.min():.2f} .. {stats.lon_max.max():.2f}  "
      f"(bins {lon_bins[0]:.2f} .. {lon_bins[-1]:.2f})")
print(f"observed lat range {stats.lat_min.min():.2f} .. {stats.lat_max.max():.2f}  "
      f"(bins {lat_bins[0]:.2f} .. {lat_bins[-1]:.2f})")
for v in VARS:
    n = stats["n_" + v].sum()
    print(f"{v:8s} n={n:,.0f}  global min={stats['min_'+v].min():.4f}  "
          f"mean={stats['sum_'+v].sum()/n:.4f}  max={stats['max_'+v].max():.4f}")


## Monthly mean per bin -> zarr

In [ ]:
OUTPUTS = {"z":    ("z",    "mean_z_month_v3.zarr"),
           "temp": ("temp", "mean_temp_month_v3.zarr"),
           "sal":  ("sal",  "mean_sal_month_v3.zarr")}
def to_da(v, name):
    mean = SUM[v] / np.where(COUNT[v] > 0, COUNT[v], np.nan)
    return xr.DataArray(mean, dims=("start_month", "lon", "lat"),
                        coords={"start_month": np.arange(1, 13), "lon": lon_c, "lat": lat_c},
                        name=name)

means = {}
for v, (name, fname) in OUTPUTS.items():
    da = to_da(v, name)
    ds = da.to_dataset()
    ds["count"] = xr.DataArray(COUNT[v], dims=("start_month", "lon", "lat"),
                               coords=ds.coords)
    ds.to_zarr(fname, mode="w")
    means[v] = ds
    finite = np.isfinite(da.values)
    print(f"{fname}: {dict(ds.sizes)}  filled bins={finite.sum()}/{finite.size}  "
          f"bin-mean min={np.nanmin(da.values):.4f} median={np.nanmedian(da.values):.4f} "
          f"max={np.nanmax(da.values):.4f}")


## Plots — one 3x4 monthly grid per variable

In [ ]:
labs = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']

def plot_monthly(da, cmap, label, name, levels=None):
    """3x4 monthly grid, same style as Analysis/Monthly_means_z_temp_sal.ipynb."""
    proj = ccrs.PlateCarree()
    fig, ax = plt.subplots(4, 3, figsize=(12, 10), subplot_kw=dict(projection=proj))
    ax = ax.ravel()
    if levels is None:
        vmin = float(np.nanpercentile(da.values, 1)); vmax = float(np.nanpercentile(da.values, 99))
        levels = np.linspace(vmin, vmax, 13)
    levels = np.asarray(levels, dtype=float)
    ncol = plt.get_cmap(cmap).N if isinstance(cmap, str) else cmap.N
    norm = BoundaryNorm(levels, ncolors=ncol, clip=True)

    for i in range(12):
        dd = da.sel(start_month=i + 1)
        pcm = dd.plot.contourf(x="lon", y="lat", ax=ax[i], transform=proj, cmap=cmap,
                               levels=levels, norm=norm, add_colorbar=False)
        ax[i].coastlines()
        ax[i].add_feature(cf.LAND, facecolor="#8e9497ff", zorder=0)
        ax[i].set_title(labs[i], fontsize=10)
        ax[i].set_xticks(np.arange(np.floor(dd.lon.min()), np.ceil(dd.lon.max()) + 1e-6, 10), crs=proj)
        ax[i].set_yticks(np.arange(np.floor(dd.lat.min()), np.ceil(dd.lat.max()) + 1e-6, 5), crs=proj)
        ax[i].xaxis.set_major_formatter(LongitudeFormatter(number_format=".0f", degree_symbol="\u00b0"))
        ax[i].yaxis.set_major_formatter(LatitudeFormatter(number_format=".0f", degree_symbol="\u00b0"))
        ax[i].tick_params(labelsize=8)
        ax[i].set_xlabel(""); ax[i].set_ylabel("")

    fig.subplots_adjust(left=0.05, right=0.95, bottom=0.08, top=0.90, wspace=0.05, hspace=0.4)
    cbar = fig.colorbar(pcm, ax=ax, orientation="horizontal", fraction=0.05, pad=0.10)
    cbar.ax.set_xlabel(label, fontsize=10)
    out = os.path.join(fig_dir, f"v2.mean_{name}_LPT_v3.png")
    plt.savefig(out, dpi=200, bbox_inches="tight")
    print("saved", out)
    plt.show()


In [ ]:
mean_z    = xr.open_zarr("mean_z_month_v3.zarr")
mean_temp = xr.open_zarr("mean_temp_month_v3.zarr")
mean_sal  = xr.open_zarr("mean_sal_month_v3.zarr")

plot_monthly(mean_z.z, "turbo", "mean depth [m]", "z",
             levels=[0, 50, 100, 150, 200, 300, 400, 500])

In [ ]:
plot_monthly(mean_temp.temp, cm.cm.balance, "mean temperature [\u00b0C]", "temp",
             levels=[0, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30])

In [ ]:
plot_monthly(mean_sal.sal, "jet", "mean salinity", "sal",
             levels=np.insert(np.arange(30, 36, 0.5), 0, 0))

In [ ]:
# particle counts per release month (number of binned samples, obs>0)
cnt = mean_z["count"].sum(("lon", "lat")).values
for m in range(12):
    print(f"{labs[m]}  stores={int((stats.month==m+1).sum()):4d}  binned samples={cnt[m]:,.0f}")
print(f"TOTAL binned samples: {cnt.sum():,.0f}")